# 66. Decoupled BO：評価費用・Mixed・バッチ・Multi-Fidelity

Notebook 65で学んだ部分観測を、測定工程の費用や設計空間の制約に拡張します。各例は候補点と評価マスクを返します。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.models import ModelListGP, SingleTaskGP, MixedSingleTaskGP, SingleTaskMultiFidelityGP
from botorch.fit import fit_gpytorch_mll
from botorch.sampling.list_sampler import ListSampler
from botorch.sampling.normal import SobolQMCNormalSampler
from gpytorch.mlls import SumMarginalLogLikelihood
from robotorchan.acquisition.factory import create_acquisition
from robotorchan.optim.decoupled import optimize_acqf_decoupled
from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace
from robotorchan.workflow.cost_budget import CostBudget

torch.manual_seed(0)
torch.set_num_threads(1)
bounds = torch.tensor([[0.], [1.]], dtype=torch.double)
grid = torch.linspace(0, 1, 32, dtype=torch.double).unsqueeze(-1)
masks = torch.tensor([[True, False], [False, True], [True, True]])
costs = bounds.new_tensor([1., 3.])
def fit(models):
    model = ModelListGP(*models)
    fit_gpytorch_mll(SumMarginalLogLikelihood(model.likelihood, model))
    model.eval()
    return model

def truth(x):
    return torch.cat((1-(x-.7)**2, 1-(x-.3)**2), dim=-1)

X = grid[[0, 10, 20, 31]]
Y = truth(X)
model = fit([SingleTaskGP(X, Y[:, j:j+1]) for j in range(2)])

def epig(model, target):
    return create_acquisition(
        "qDecoupledExpectedPredictiveInformationGain", model=model,
        target_X=target, X_evaluation_mask=masks[:1],
        observation_noise=bounds.new_tensor([.01, .01]),
    )

def sobol(acq, b, **kwargs):
    return optimize_acqf_decoupled(
        acq, b, masks, optimizer="sobol",
        optimizer_options={"num_samples": 32}, seed=0, **kwargs,
    )

acq = epig(model, grid)

## 1. 固定費用と工程共有費用

出力別費用は加算されます。一方、同時測定で共通前処理を共有するなら、マスクごとの費用を直接指定します。予算による候補の除外はoptimizerの責務ですが、費用の記録は実験側の責務です。

In [ ]:
ledger = CostBudget(20, spent=16)
for label, kwargs in [
    ("additive", {"output_costs": costs}),
    ("shared setup", {"mask_costs": bounds.new_tensor([2., 4., 4.5])}),
]:
    x, a, v = sobol(acq, bounds, budget=ledger, **kwargs)
    print(label, x.tolist(), a.tolist(), float(v))
assert ledger.spent == 16

## 2. 入力依存費用

設計値が大きいほど測定費用が増える例です。cost_functionは各t-batchに対して正の費用を返します。ここでは費用効率のみを最適化し、入力依存hard budgetとは区別します。

In [ ]:
def predicted_cost(x, mask):
    base = (mask.to(x) @ costs).sum()
    return base + x[..., 0].sum(dim=-1)

x, a, v = sobol(acq, bounds, cost_function=predicted_cost)
print(x, a, v, predicted_cost(x, a))
plt.plot(grid[:, 0], predicted_cost(grid.unsqueeze(-2), masks[:1]), label="Output 1")
plt.plot(grid[:, 0], predicted_cost(grid.unsqueeze(-2), masks[1:2]), label="Output 2")
plt.xlabel("Design")
plt.ylabel("Predicted cost")
plt.legend()
plt.show()

## 3. q-batchと候補制約

q=2では二つの設計点と二つの評価マスクを選びます。jointはマスク割当を同時に比較します。候補制約は入力座標の制約であり、未知の品質制約とは異なります。

In [ ]:
x, a, v = sobol(acq, bounds, q=2, output_costs=costs, budget=ledger)
print("Joint:", x, a, v)
assert x.shape == (2, 1) and a.shape == (2, 2)
assert float((a.to(costs) @ costs).sum()) <= ledger.remaining
constraint = CandidateConstraints(
    inequality_constraints=((torch.tensor([0]), bounds.new_tensor([1.]), .3),)
)
x, a, v = optimize_acqf_decoupled(
    acq, bounds, masks, constraints=constraint,
    num_restarts=2, raw_samples=16, options={"maxiter": 20},
)
print("Constrained:", x, a, v)
assert (x >= .3 - 1e-6).all()

## 4. Mixed変数

連続温度、整数工程回数、カテゴリレシピを区別します。MixedSingleTaskGPはカテゴリ用カーネルを持ち、MixedVariableSpaceは候補の合法値を管理します。整数は数値座標としてモデル化します。

In [ ]:
mixed_bounds = bounds.new_tensor([[0, 1, 0], [1, 3, 1]])
mx = bounds.new_tensor([[.1, 1, 0], [.4, 2, 0], [.8, 3, 0],
                        [.2, 3, 1], [.5, 1, 1], [.9, 2, 1]])
def mixed_truth(x):
    t, n, recipe = x.unbind(-1)
    return torch.stack((1-(t-.7)**2-.1*(n-2)**2-.1*recipe,
                        1-(t-.3)**2-.05*(n-1)**2+.1*recipe), dim=-1)

my = mixed_truth(mx)
mixed_model = fit([MixedSingleTaskGP(mx, my[:, j:j+1], cat_dims=[2]) for j in range(2)])
space = MixedVariableSpace(mixed_bounds, integer_dims=(1,), categorical_values={2: (0., 1.)})
target = torch.cartesian_prod(grid[::4, 0], bounds.new_tensor([1, 2, 3]), bounds.new_tensor([0, 1]))
x, a, v = sobol(epig(mixed_model, target), mixed_bounds, variable_space=space, output_costs=costs)
print(x, a, v)
assert x[0, 1] == x[0, 1].round()
assert x[0, 2] in (0., 1.)

## 5. Multi-Fidelity：測定対象と忠実度の選択

低忠実度0.25と目標忠実度1.0を比較します。忠実度は入力の第2列です。KGの終端価値は目標忠実度で評価し、測定費用は忠実度別に変えます。

In [ ]:
mf_bounds = bounds.new_tensor([[0, .25], [1, 1]])
mf_x = torch.cartesian_prod(grid[::10, 0], bounds.new_tensor([.25, 1.]))
def mf_truth(x):
    return truth(x[..., :1]) - (1-x[..., 1:2])*bounds.new_tensor([.3, .2])

mf_y = mf_truth(mf_x)
mf_model = fit([SingleTaskMultiFidelityGP(mf_x, mf_y[:, j:j+1], data_fidelities=[1]) for j in range(2)])
target = torch.cat((grid, torch.ones_like(grid)), dim=-1)
def mf_kg(model):
    return create_acquisition(
        "qDecoupledKnowledgeGradient", model=model, decision_X=target,
        weights=bounds.new_tensor([.5, .5]),
        sampler=ListSampler(*(SobolQMCNormalSampler(torch.Size([4]), seed=j) for j in range(2))),
        X_evaluation_mask=masks[:1], target_fidelities={1: 1.0},
    )

mf_budget = CostBudget(8)
results = []
for fidelity, multiplier in [(0.25, 1), (1., 4)]:
    x, a, v = sobol(mf_kg(mf_model), mf_bounds, fixed_features={1: fidelity},
                    output_costs=costs*multiplier, budget=mf_budget)
    results.append((float(v), x, a, multiplier))
    print(fidelity, x, a, v)
score, x, a, multiplier = max(results, key=lambda row: row[0])
mf_budget.record(float((a.to(costs) @ (costs*multiplier)).sum()))
print("Selected:", x, a, "spent:", mf_budget.spent)
with torch.no_grad():
    for fidelity in [.25, 1.]:
        points = torch.cat((grid, torch.full_like(grid, fidelity)), dim=-1)
        p = mf_model.posterior(points)
        plt.plot(grid[:, 0], p.mean[:, 0], label=f"fidelity={fidelity}")
plt.xlabel("Design")
plt.ylabel("Output 1 posterior mean")
plt.legend()
plt.show()

## 6. 評価予算付き反復と比較

既存ベンチマークを使って、低忠実度を許す場合と目標忠実度だけの場合を同じ予算で比較します。少数seedの動作例であり、性能優位性を結論づけるものではありません。

In [ ]:
from robotorchan.benchmarks.decoupled_multifidelity import run_multifidelity_decoupled_benchmark
for multifidelity in [False, True]:
    rows = run_multifidelity_decoupled_benchmark(
        budget=36, n_initial=2, num_samples=4, multifidelity=multifidelity,
    )
    plt.step([r["cost"] for r in rows], [r["simple_regret"] for r in rows],
             where="post", label=rows[0]["method"])
plt.xlabel("Cumulative cost")
plt.ylabel("Simple regret")
plt.legend()
plt.show()

## 7. Multi-Fidelity Hypervolume Knowledge Gradient

固定重みKGと異なり、目標忠実度でのPareto推奨集合の価値を評価します。ここではBoTorch nativeのone-shot獲得関数を使用し、この例は費用補正なしの候補提案です。補助点は測定対象に含まれません。

In [ ]:
mf_hvkg = create_acquisition(
    "qMultiFidelityHypervolumeKnowledgeGradient",
    model=mf_model, target_fidelities={1: 1.0},
    ref_point=bounds.new_tensor([0., 0.]),
    num_fantasies=1, num_pareto=1,
    X_evaluation_mask=masks[:1],
)
x_hv, a_hv, v_hv = optimize_acqf_decoupled(
    mf_hvkg, mf_bounds, masks,
    num_restarts=2, raw_samples=8,
    options={"maxiter": 20},
)
assert x_hv.shape == (1, 2) and a_hv.shape == (1, 2)
assert torch.isfinite(v_hv)
print("MFHVKG:", x_hv, a_hv, v_hv)